# Lab 4 · Exercise A — One toy, three ways (ROOT Team course)
We use the course `02-roofit` of the ROOT Project (`third_party/root-cpp-course-roofit/`, LGPL, see `ORIGIN.md`).
One weighted toy simulator, **one dataset, three fits**: unbinned analytic shapes, RDataFrame + `Vary` templates + HistFactory, likelihood anatomy.
The macros are C++ (modern ROOT, run just in time): this notebook runs them and analyses the results in Python.

In [ ]:
import ROOT
%jsroot on   # interactive drawing in the notebook (needed on Binder)
import numpy as np

ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## A1 · Run the pipeline
Each macro takes a few seconds to a minute.

In [ ]:
import subprocess, pathlib
WD = pathlib.Path("../../third_party/root-cpp-course-roofit").resolve()

def run(macro):
    r = subprocess.run(["root", "-b", "-l", "-q", macro], cwd=WD, capture_output=True, text=True)
    print("=== %s (exit %d) ===" % (macro, r.returncode))
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode

codes = [run(m) for m in ["make_toys.cpp", "unbinned.cpp", "analysis.cpp", "histfactory.cpp", "likelihood.cpp"]]

In [ ]:
check("all macros ran", all(c == 0 for c in codes))
for f in ["data.root", "sim.root", "hists.root", "results/meas_combined_meas_model.root"]:
    check("produced " + f, (WD / f).exists())

## A2 · Where do $\\mu$ and $\\theta$ act?
Read `physics.h` (the “theory”). Answer in this cell:
1. Which function applies $\\mu$ and $\\theta$ to the simulated events? Is the simulation ever re-run?
2. What are `kThetaNom`, `kThetaDelta`, `kThetaTrue`? Which of them does the analysis know, and which only nature?

In [ ]:
for i, line in enumerate((WD / "physics.h").read_text().splitlines(), 1):
    if "theta" in line.lower():
        print(i, line)

## A3 · The three fits
Open the HistFactory workspace, fit, and read $\\hat\\mu$ and $\\hat\\theta=\\theta_{\\rm nom}+\\hat\\alpha\\,\\Delta\\theta$.
Compare with the truth ($\\mu_{\\rm true}=1$, $\\theta_{\\rm true}=1.3$).

In [ ]:
f  = ROOT.TFile.Open(str(WD / "results/meas_combined_meas_model.root"))
w  = f.Get("combined")
pdf, data = w.pdf("simPdf"), w.data("obsData")
mu, alpha = w.var("mu"), w.var("alpha_theta")

res = pdf.fitTo(data, ROOT.RooFit.Save(), ROOT.RooFit.PrintLevel(-1))
THETA_NOM, THETA_DELTA = 1.0, 0.4

theta_hat = THETA_NOM + alpha.getVal() * THETA_DELTA
theta_err = alpha.getError() * THETA_DELTA
print("mu = %.3f +- %.3f" % (mu.getVal(), mu.getError()))
print("theta = %.3f +- %.3f   (true 1.3)" % (theta_hat, theta_err))

In [ ]:
check("fit converged", res.status() == 0)
check("mu compatible with the truth (1) within 3 sigma", abs(mu.getVal() - 1) < 3 * mu.getError())
check("theta compatible with the truth (1.3) within 3 sigma", abs(theta_hat - 1.3) < 3 * theta_err)

## A4 · Slice vs profile of $\\mu$ (as in Lab 2)

In [ ]:
nll = pdf.createNLL(data)
frame = mu.frame(ROOT.RooFit.Range(mu.getVal() - 4 * mu.getError(), mu.getVal() + 4 * mu.getError()))
nll.plotOn(frame, ROOT.RooFit.ShiftToZero(), ROOT.RooFit.LineColor(ROOT.kBlack))
nll.createProfile(ROOT.RooArgSet(mu)).plotOn(frame, ROOT.RooFit.ShiftToZero(), ROOT.RooFit.LineColor(ROOT.kRed))
c = ROOT.TCanvas("c", "c", 700, 500)
frame.SetMinimum(0); frame.SetMaximum(3)
frame.Draw(); c.Draw()
print("corr(mu, alpha_theta) = %.2f" % res.correlation("mu", "alpha_theta"))

## Questions (3–4 lines each)
1. The unbinned fit (`unbinned.cpp`) uses the *known* shapes, the HistFactory fit uses *templates*. Which has the smaller uncertainty on the signal, and why?
2. How does the nuisance parameter $\\theta$ change the uncertainty on $\\mu$ (slice vs profile)?
3. In `histfactory.cpp`, what does `ActivateStatError` add? Which real-analysis effect does it model?

**Solution notes (published after the deadline):**
1. Known shapes use the information in the continuous mass variable and need no templates, so they are usually more precise. Templates pay a price for binning and for the finite MC.
2. The profile is wider than the slice: the unknown background tilt $\\theta$ is partly degenerate with the signal normalisation.
3. A Poisson or Gaussian nuisance parameter per bin for the finite MC statistics of the templates.